# Sistema Multi-Agente de Saúde com Memória Episódica

## Introdução

Este notebook demonstra como implementar um **sistema multi-agente de saúde com memória episódica** usando o AgentCore Memory SDK e os memory hooks do Strands. Esta abordagem fornece gerenciamento automático de memória sem chamadas manuais à API.

### Detalhes do Tutorial

| Informação          | Detalhes                                                                         |
|:--------------------|:----------------------------------------------------------------------------------|
| Tipo do tutorial    | Memória Episódica com Coordenação Multi-Agente                                   |
| Tipo do agente      | Sistema Assistente de Saúde                                                      |
| Framework agêntico  | Strands Agents com Memory Hooks                                                  |
| Modelo LLM          | Anthropic Claude Sonnet 4                                                        |
| Componentes         | Memória Episódica, Memory Hooks, Integração com HealthLake                       |
| Complexidade        | Intermediário                                                                    |

Você aprenderá:

- Como usar o MemoryClient SDK para memória episódica
- Criar memory hooks para gerenciamento automático de memória
- Implementar agentes especializados com memória episódica compartilhada
- Integrar consultas FHIR em tempo real do HealthLake

## Como a Memória Episódica Ajuda Este Assistente de Saúde

A **EpisodicStrategy** captura interações como episódios estruturados e gera insights significativos entre sessões. Isso vai além de registrar "o que aconteceu" para entender "por quê" e "como" as interações se desenrolaram.

### Processo em Três Passos

1. **Extração** – Identifica insights úteis da memória de curto prazo (eventos) e os coloca na memória de longo prazo como episódios estruturados
2. **Consolidação** – Determina se deve gravar informações em um novo episódio ou atualizar um existente
3. **Reflexão** – Gera insights entre múltiplos episódios para identificar padrões e melhorias

### Estrutura do Episódio

Cada episódio captura:
- **Situação**: O que o profissional de saúde estava tentando realizar
- **Intenção**: O objetivo principal da interação
- **Avaliação**: Se o objetivo foi alcançado com sucesso
- **Justificativa**: Por que a avaliação foi feita
- **Análise turno a turno**: Detalhamento mostrando roteamento de agentes, uso de ferramentas e tomada de decisão
- **Reflexão em nível de episódio**: Insights sobre o que funcionou bem nesta sessão específica

### Reflexões em Nível de Paciente

Reflexões consolidam múltiplos episódios para extrair insights mais amplos:
- **Estratégias bem-sucedidas**: Padrões que funcionam consistentemente (ex.: protocolo de roteamento, apresentação de dados)
- **Casos de uso comuns**: Tipos de consultas frequentemente feitas para este paciente
- **Melhorias potenciais**: Áreas onde o assistente poderia ser mais eficaz
- **Lições aprendidas**: Insights que abrangem múltiplas interações

### Benefícios para Fluxos de Trabalho em Saúde

1. **Roteamento aprimorado**: Aprenda qual agente lida com quais tipos de perguntas de forma mais eficaz
2. **Melhor apresentação de dados**: Entenda como formatar dados complexos de saúde para compreensão rápida
3. **Reconhecimento de padrões**: Identifique padrões comuns de consulta para pacientes específicos
4. **Melhoria de qualidade**: Acompanhe o que funciona e o que não funciona em múltiplas sessões
5. **Consciência contextual**: Interações futuras se beneficiam das lições aprendidas em sessões passadas

Neste tutorial, você verá como os episódios capturam o fluxo completo de interações multi-agente, e como as reflexões fornecem insights acionáveis para melhorar o assistente de saúde ao longo do tempo.

---
## Contexto do Cenário

Criaremos um **Sistema Assistente de Saúde** com:
1. Um **Agente Supervisor** que roteia perguntas de pacientes
2. Um **Agente de Sinistros** para seguros e faturamento
3. Um **Agente de Dados Demográficos** para informações do paciente
4. Um **Agente de Medicamentos** para prescrições

Todos os agentes usam memory hooks para salvar automaticamente conversas na memória episódica.

## Arquitetura
<div style="text-align:left">
    <img src="architecture.png" width="75%" />
</div>

## Pré-requisitos

- Python 3.10+
- Credenciais AWS com permissões para Bedrock e AgentCore Memory
- Amazon HealthLake datastore (opcional)

Vamos começar!

## Passo 1: Configuração do Ambiente
Vamos começar importando todas as bibliotecas necessárias e definindo os clientes para fazer este notebook funcionar.

In [ ]:
%pip install -qr ./requirements.txt

In [ ]:
import logging
from datetime import datetime
from botocore.exceptions import ClientError
from strands import Agent, tool
from strands.hooks import HookProvider, HookRegistry
from bedrock_agentcore.memory import MemoryClient

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("healthcare-assistant")

Vamos definir as entradas do usuário para a Configuração de Memória

In [ ]:
MEMORY_NAME = "healthcare_episodic_memory"
PATIENT_ID = "b2055b4d-ac17-4d94-8c5b-3395e4c334dd"
region = "us-east-1"  # Replace with your AWS region
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}"
MODEL_ID = (
    "global.anthropic.claude-sonnet-4-20250514-v1:0"  # Replace with your Model ID
)

print("Memory Configuration:")
print(f"  Memory Name: {MEMORY_NAME}")
print(f"  Patient ID: {PATIENT_ID}")
print(f"  Region: {region}")
print(f"  Session ID: {SESSION_ID}")
print(f"  Model ID: {MODEL_ID}")

## Passo 2: Configurar o HealthLake Datastore

Configure o HealthLake FHIR datastore com dados de pacientes para os agentes de saúde consultarem.

In [ ]:
import boto3
import requests
import time
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest

# HealthLake configuration
HEALTHLAKE_REGION = (
    input("Enter HealthLake region (or press Enter for us-east-1): ").strip()
    or "us-east-1"
)
DATASTORE_ID = input(
    "Enter HealthLake datastore ID (or press Enter to create new): "
).strip()

healthlake_client = boto3.client("healthlake", region_name=HEALTHLAKE_REGION)

# Create new datastore if not provided
if not DATASTORE_ID:
    create_new = (
        input(
            "\nNo datastore ID provided. Create new HealthLake datastore with Synthea data? (yes/no): "
        )
        .strip()
        .lower()
    )

    if create_new == "yes":
        print("\nCreating HealthLake datastore...")

        # Create datastore
        create_response = healthlake_client.create_fhir_datastore(
            DatastoreName=f"healthcare-demo-{int(time.time())}",
            DatastoreTypeVersion="R4",
            PreloadDataConfig={"PreloadDataType": "SYNTHEA"},
        )

        DATASTORE_ID = create_response["DatastoreId"]
        print(f"✅ Datastore created: {DATASTORE_ID}")
        print(
            "⏳ Waiting for datastore to become ACTIVE (this may take 10-15 minutes)..."
        )

        # Wait for ACTIVE status
        while True:
            status_response = healthlake_client.describe_fhir_datastore(
                DatastoreId=DATASTORE_ID
            )
            status = status_response["DatastoreProperties"]["DatastoreStatus"]

            if status == "ACTIVE":
                print("✅ Datastore is ACTIVE")
                break
            elif status in ["FAILED", "DELETING"]:
                print(f"❌ Datastore creation failed with status: {status}")
                raise Exception(f"Datastore creation failed: {status}")

            print(f"   Status: {status}...")
            time.sleep(30)

        print(f"\n✅ Synthea data loaded. Using default patient ID: {PATIENT_ID}")


# Get HealthLake endpoint
datastore = healthlake_client.describe_fhir_datastore(DatastoreId=DATASTORE_ID)
HEALTHLAKE_ENDPOINT = datastore["DatastoreProperties"]["DatastoreEndpoint"]


def query_healthlake(resource_type, search_params=None, resource_id=None):
    """Query HealthLake FHIR API"""
    if resource_id:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}/{resource_id}"
    else:
        url = f"{HEALTHLAKE_ENDPOINT}/{resource_type}"
        if search_params:
            params = "&".join([f"{k}={v}" for k, v in search_params.items()])
            url += f"?{params}"

    session = boto3.Session()
    credentials = session.get_credentials()

    request = AWSRequest(
        method="GET", url=url, headers={"Accept": "application/fhir+json"}
    )
    SigV4Auth(credentials, "healthlake", HEALTHLAKE_REGION).add_auth(request)

    response = requests.get(url, headers=dict(request.headers))

    if response.status_code == 200:
        return response.json()
    else:
        return {"error": f"Failed to fetch: {response.text}"}


print(f"\n{'=' * 70}")
print("HealthLake Configuration:")
print(f"  Datastore ID: {DATASTORE_ID}")
print(f"  Endpoint:     {HEALTHLAKE_ENDPOINT}")
print(f"  Region:       {HEALTHLAKE_REGION}")
print(f"  Patient ID:   {PATIENT_ID}")
print(f"{'=' * 70}")

## Passo 3: Criar Memória com Estratégia Episódica

Criaremos um único recurso de memória que suportará múltiplas branches - uma para cada agente. Este recurso de memória compartilhado atua como a base, enquanto as branches fornecem contextos isolados para as conversas de cada agente.

Pense nisso como um repositório Git: um repositório (recurso de memória) com múltiplas branches (contextos de agentes).

In [ ]:
client = MemoryClient(region_name=region)

strategies = [
    {
        "episodicMemoryStrategy": {
            "name": "HealthcareEpisodes",
            "description": "Captures healthcare interactions as episodes",
            "namespaces": ["healthcare/{actorId}/{sessionId}/"],
            "reflectionConfiguration": {"namespaces": ["healthcare/{actorId}/"]},
        }
    }
]

try:
    memory = client.create_memory_and_wait(
        name=MEMORY_NAME,
        strategies=strategies,
        description="Healthcare system with episodic memory",
        event_expiry_days=7,  # Short-term conversation expires after 7 days
        max_wait=300,
        poll_interval=10,
    )
    memory_id = memory["id"]
    logger.info(f"Memory created successfully with ID: {memory_id}")
except ClientError as e:
    if e.response["Error"]["Code"] == "ValidationException" and "already exists" in str(
        e
    ):
        # If memory already exists, retrieve its ID
        memories = client.list_memories()
        memory_id = next(
            (m["id"] for m in memories if m["id"].startswith(MEMORY_NAME)), None
        )
        logger.info(f"Memory already exists. Using existing memory: {memory_id}")
except Exception as e:
    # Handle any errors during memory creation
    print(f"❌ ERROR: {e}")
    import traceback

    traceback.print_exc()

    # Cleanup on error - delete the memory if it was partially created
    if memory_id:
        try:
            client.delete_memory_and_wait(memory_id=memory_id)
            logger.info(f"Cleaned up memory: {memory_id}")
        except Exception as cleanup_error:
            logger.info(f"Failed to clean up memory: {cleanup_error}")

### Entendendo o Memory Branching para Sistemas Multi-Agente de Saúde

O recurso de memória que criamos suporta **branching** - uma funcionalidade crítica para arquiteturas multi-agente de saúde. Veja como funciona:

**Recurso de Memória Único, Múltiplas Branches:**
- Todos os agentes compartilham o mesmo `memory_id` e `session_id`
- Cada agente recebe seu próprio `branch_name` para contexto isolado

**Benefícios Principais para Sistemas Multi-Agente de Saúde:**

1. **Isolamento de Contexto**: Cada agente mantém seu próprio histórico de conversa sem interferência
   - O agente de sinistros só vê conversas sobre seguros e faturamento
   - O agente de dados demográficos só vê conversas sobre informações do paciente
   - O agente de medicamentos só vê conversas relacionadas a prescrições
   - O supervisor vê o fluxo principal de roteamento e coordenação

2. **Segurança de Execução Paralela**: Múltiplos agentes podem executar simultaneamente
   - Sem conflitos de memória quando agentes rodam em paralelo
   - Cada branch é acessível independentemente
   - Crítico para fluxos de trabalho em saúde que requerem processamento concorrente

3. **Trilha de Auditoria Clara**: As interações de cada agente são rastreáveis
   - Inspecione o que cada agente de saúde discutiu
   - Depure problemas específicos de cada agente
   - Entenda o fluxo das conversas de cuidado ao paciente
   - Mantenha conformidade e requisitos de documentação

**Estrutura de Branches para Saúde:**
- Branch `main`: Decisões de roteamento do supervisor
- Branch `claims_agent`: Conversas sobre seguros e faturamento
- Branch `demographics_agent`: Atualizações de informações do paciente
- Branch `medication_agent`: Discussões sobre prescrições

## Passo 4: Criar Memory Hook Provider com Suporte a Branches

In [ ]:
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from strands.hooks import AgentInitializedEvent, MessageAddedEvent
from bedrock_agentcore.memory import MemorySessionManager


class HealthcareMemoryHooks(HookProvider):
    def __init__(
        self, memory_id: str, region_name: str = None, branch_name: str = "main"
    ):
        """Initialize the hook with a MemorySessionManager.

        Args:
            memory_id: The AgentCore Memory ID
            region_name: AWS region for the memory service
            branch_name: Branch name for this agent's memory (default: "main")
        """
        if region_name is None:
            region_name = region  # Use global region variable

        self.memory_manager = MemorySessionManager(
            memory_id=memory_id, region_name=region_name
        )
        self.memory_id = memory_id
        self.branch_name = branch_name
        self._sessions = {}  # Cache session objects per actor/session combo
        self._branch_initialized = False  # Track if branch has been created

    def _get_or_create_session(self, actor_id: str, session_id: str):
        """Get or create a MemorySession for the given actor/session."""
        key = f"{actor_id}:{session_id}"
        if key not in self._sessions:
            self._sessions[key] = self.memory_manager.create_memory_session(
                actor_id=actor_id, session_id=session_id
            )
        return self._sessions[key]

    def _initialize_branch(self, actor_id: str, session_id: str):
        """Initialize a branch if it doesn't exist and this is not the main branch."""
        if self._branch_initialized or self.branch_name == "main":
            return

        try:
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Check if branch already exists
            branches = memory_session.list_branches()
            branch_exists = any(b.name == self.branch_name for b in branches)

            if not branch_exists:
                # Get the last event from main branch to fork from
                main_events = memory_session.list_events(branch_name="main")
                if not main_events:
                    # Create initial event in main branch
                    memory_session.add_turns(
                        [
                            ConversationalMessage(
                                "Healthcare system initialized", MessageRole.ASSISTANT
                            )
                        ]
                    )
                    main_events = memory_session.list_events(branch_name="main")

                if main_events:
                    last_event = main_events[-1]
                    # Create the branch
                    memory_session.fork_conversation(
                        root_event_id=last_event.eventId,
                        branch_name=self.branch_name,
                        messages=[
                            ConversationalMessage(
                                f"Starting {self.branch_name} healthcare branch",
                                MessageRole.ASSISTANT,
                            )
                        ],
                    )
                    logger.info(f"✅ Created healthcare branch: {self.branch_name}")

            self._branch_initialized = True

        except Exception as e:
            logger.error(
                f"Failed to initialize healthcare branch {self.branch_name}: {e}"
            )

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when healthcare agent starts"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning(
                    "Missing actor_id or session_id in healthcare agent state"
                )
                return

            # Initialize branch if needed (for non-main branches)
            if self.branch_name != "main":
                self._initialize_branch(actor_id, session_id)

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Get last 5 conversation turns from this branch
            recent_turns = memory_session.get_last_k_turns(
                k=5, branch_name=self.branch_name, include_parent_branches=False
            )

            if recent_turns:
                # Add context to agent's system prompt
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get("role", "unknown").lower()
                        text = message.get("content", {}).get("text", "")
                        if text:
                            context_messages.append(f"{role.title()}: {text}")

                if context_messages:
                    context = "\n".join(context_messages[-10:])  # Last 10 messages
                    event.agent.system_prompt += (
                        f"\n\nRecent healthcare conversation history:\n{context}\n\n"
                        "Continue the conversation naturally based on this context."
                    )
                    logger.info(
                        f"✅ Loaded healthcare context from branch '{self.branch_name}'"
                    )

        except Exception as e:
            logger.error(f"Failed to load healthcare conversation history: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """Store healthcare conversation turns in memory on the appropriate branch"""
        try:
            # Get session info from agent state
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")

            if not actor_id or not session_id:
                logger.warning(
                    "Missing actor_id or session_id in healthcare agent state"
                )
                return

            # Get the memory session
            memory_session = self._get_or_create_session(actor_id, session_id)

            # Get the last message
            messages = event.agent.messages
            if not messages:
                return

            last_message = messages[-1]
            role_str = last_message.get("role", "").upper()
            content_text = last_message.get("content", [{}])[0].get("text", "")

            if not content_text:
                logger.debug("Skipping empty healthcare message")
                return

            # Map role string to MessageRole enum
            role_mapping = {
                "USER": MessageRole.USER,
                "ASSISTANT": MessageRole.ASSISTANT,
                "TOOL": MessageRole.TOOL,
            }
            message_role = role_mapping.get(role_str, MessageRole.USER)

            # Store the message on the appropriate branch
            if self.branch_name == "main":
                # Main branch - just add turns normally
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)]
                )
            else:
                # Non-main branch - need to append to existing branch
                # Initialize branch if it doesn't exist
                if not self._branch_initialized:
                    self._initialize_branch(actor_id, session_id)

                # Add to existing branch
                memory_session.add_turns(
                    messages=[ConversationalMessage(content_text, message_role)],
                    branch={"name": self.branch_name},
                )

            logger.info(f"Memory saved to healthcare branch: {self.branch_name}")

        except Exception as e:
            logger.error(f"Failed to store healthcare message: {e}")

    def get_session(self, actor_id: str, session_id: str):
        """Get the memory session object for direct access."""
        return self._get_or_create_session(actor_id, session_id)

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register healthcare memory hooks with the registry."""
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)


print("✅ Healthcare memory hook provider defined")

## Passo 5: Criar Arquitetura Multi-Agente de Saúde com Memory Branching

Nesta seção, criaremos agentes de saúde especializados que usam **diferentes memory branches** para demonstrar a capacidade de branching:

### Estratégia de Branching para Saúde:
- **Branch Main**: Armazena as decisões de roteamento do supervisor e serve como a thread de conversa base
- **Branch claims_agent**: Uma branch separada para conversas sobre seguros, faturamento e sinistros
- **Branch demographics_agent**: Uma branch separada para informações e detalhes de contato do paciente
- **Branch medication_agent**: Uma branch separada para conversas sobre prescrições e medicamentos

Cada agente de saúde especializado opera em sua própria branch, que é automaticamente criada a partir da conversa principal quando usada pela primeira vez. Isso permite:

- Fluxos de conversa independentes para diferentes especializações de saúde
- Isolamento de contexto médico específico do domínio
- Preservação da thread de conversa principal do supervisor
- Conformidade com requisitos de separação de dados de saúde
- Trilhas de auditoria claras para diferentes tipos de interações com pacientes

### Papéis dos Agentes de Saúde:
- **Agente Supervisor**: Roteia perguntas de pacientes para os especialistas apropriados
- **Agente de Sinistros**: Lida com sinistros de seguros, consultas de faturamento e questões de cobertura
- **Agente de Dados Demográficos**: Gerencia informações demográficas e atualizações de contato do paciente
- **Agente de Medicamentos**: Processa perguntas sobre prescrições, informações de dosagem e gerenciamento de medicamentos

Esta arquitetura garante que conversas sensíveis de saúde permaneçam devidamente isoladas, mantendo uma experiência coesa de cuidado ao paciente.

### Criando Agentes com Memória Ramificada

A seguir, definiremos system prompts e criaremos agentes que usam diferentes memory branches. Observe como usamos o mesmo `actor_id` e `session_id`, mas valores diferentes de `branch_name` para criar contextos de conversa isolados:

In [ ]:
# System prompt for the healthcare supervisor
SUPERVISOR_PROMPT = """You are a healthcare supervisor agent. Route patient questions to:
    - Claims Agent: for insurance, billing, claims questions
    - Demographics Agent: for personal info, contact details  
    - Medication Agent: for prescriptions, medications, dosage
    
    Respond briefly and indicate which agent you're routing to."""

# System prompt for the claims specialist
CLAIMS_PROMPT = """You handle insurance claims. Use the get_patient_claims tool to fetch 
    current claim data from HealthLake. Answer questions about claims, billing, and coverage."""

# System prompt for the demographics specialist
DEMOGRAPHICS_PROMPT = """You handle patient demographics. Use the get_patient_demographics tool to 
    fetch current patient data from HealthLake. Answer questions about contact details and personal information."""

# System prompt for the medication specialist
MEDICATION_PROMPT = """You handle medications. Use the get_patient_medications tool to fetch 
    current medication data from HealthLake. Answer questions about prescriptions and dosages."""

## Passo 6: Configurando Ferramentas de Saúde e Memory Hooks

Primeiro, criaremos as ferramentas de dados do HealthLake e os memory hooks para nossos agentes de saúde especializados. Cada agente recebe seu próprio memory hook configurado com um nome de branch específico:
- O agente de sinistros usa a branch `claims_agent`
- O agente de dados demográficos usa a branch `demographics_agent`
- O agente de medicamentos usa a branch `medication_agent`
- O agente supervisor usa a branch `main`

Quando esses agentes de saúde são invocados:
1. O hook verifica se a branch existe
2. Se não, ele cria uma nova branch a partir da conversa principal
3. A conversa de saúde do agente é armazenada em sua branch dedicada
4. Cada agente mantém contexto isolado para privacidade dos dados do paciente
5. As ferramentas de dados do HealthLake fornecem acesso em tempo real às informações do paciente
6. A conformidade de saúde é mantida através do isolamento adequado de dados

**Ferramentas de Dados do HealthLake:**
- `get_patient_claims` - Recupera informações de sinistros de seguros e faturamento
- `get_patient_demographics` - Busca dados de contato e demográficos do paciente
- `get_patient_medications` - Obtém dados atuais de prescrições e medicamentos

**Estrutura de Memory Branches:**
- Cada agente opera independentemente com seu próprio contexto de memória
- Conversas de pacientes são devidamente isoladas por domínio de saúde
- O supervisor coordena entre agentes mantendo a separação


In [ ]:
# Initialize healthcare memory hooks as None
supervisor_hooks = None
claims_hooks = None
demographics_hooks = None
medication_hooks = None

In [ ]:
@tool
def get_patient_claims(patient_id: str = PATIENT_ID) -> dict:
    """Get patient insurance claims from HealthLake"""
    return query_healthlake("Claim", {"patient": patient_id})


@tool
def get_patient_medications(patient_id: str = PATIENT_ID) -> dict:
    """Get patient medications from HealthLake"""
    return query_healthlake("MedicationRequest", {"patient": patient_id})


@tool
def get_patient_demographics(patient_id: str = PATIENT_ID) -> dict:
    """Get patient demographic information from HealthLake"""
    return query_healthlake("Patient", resource_id=patient_id)


# Create memory hooks for each agent
supervisor_hooks = HealthcareMemoryHooks(memory_id, region, "main")
claims_hooks = HealthcareMemoryHooks(memory_id, region, "claims_agent")
demographics_hooks = HealthcareMemoryHooks(memory_id, region, "demographics_agent")
medication_hooks = HealthcareMemoryHooks(memory_id, region, "medication_agent")

print("✅ HealthLake tools and memory hooks created")

### Criando Agentes de Saúde

Agora criaremos os agentes de saúde usando as ferramentas e memory hooks que configuramos:

- **Agente Supervisor**: Roteia perguntas (branch main)
- **Agente de Sinistros**: Seguros e faturamento (branch claims_agent)
- **Agente de Dados Demográficos**: Informações do paciente (branch demographics_agent)
- **Agente de Medicamentos**: Prescrições (branch medication_agent)

Cada agente recebe seu system prompt especializado, ferramentas relevantes e memory hooks para conversas isoladas.

In [ ]:
# Create specialized healthcare agents with memory branching
supervisor = Agent(
    model=MODEL_ID,
    system_prompt=SUPERVISOR_PROMPT,
    hooks=[supervisor_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

claims_agent = Agent(
    model=MODEL_ID,
    system_prompt=CLAIMS_PROMPT,
    tools=[get_patient_claims],
    hooks=[claims_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

demographics_agent = Agent(
    model=MODEL_ID,
    system_prompt=DEMOGRAPHICS_PROMPT,
    tools=[get_patient_demographics],
    hooks=[demographics_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

medication_agent = Agent(
    model=MODEL_ID,
    system_prompt=MEDICATION_PROMPT,
    tools=[get_patient_medications],
    hooks=[medication_hooks],
    state={"actor_id": PATIENT_ID, "session_id": SESSION_ID},
)

print("✅ Healthcare agents created with HealthLake tools and memory branching")

#### Seu Sistema Multi-Agente de Saúde com Memória Episódica está pronto!!

## Vamos testar o Assistente de Saúde.

Vamos testar nosso sistema multi-agente de saúde com um cenário de cuidado ao paciente:

**Perguntas de exemplo para tentar:**
- "Qual é o status dos meus sinistros de seguro?"
- "Você pode atualizar minhas informações de contato?"
- "Quais medicamentos estou tomando atualmente?"
- "Tenho alguma pendência de faturamento?"
- "Qual é o meu endereço atual no cadastro?"
- "Existem interações medicamentosas com minhas prescrições?"
- "Quanto devo pela minha consulta recente?"
- "Pode me informar sobre os detalhes da minha cobertura?"

In [ ]:
# Interactive chat with healthcare agents
print("Healthcare Assistant - Type 'quit' to exit\n")

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in ["quit", "exit", "q"]:
        break

    if not user_input:
        continue

    # Supervisor handles routing
    routing = str(supervisor(user_input))
    print(f"\nSupervisor: {routing}")

    # Route to appropriate agent based on supervisor's decision
    if "claims agent" in routing.lower():
        response = str(claims_agent(user_input))
        print(f"\nClaims Agent: {response}\n")
    elif "demographics agent" in routing.lower():
        response = str(demographics_agent(user_input))
        print(f"\nDemographics Agent: {response}\n")
    elif "medication agent" in routing.lower():
        response = str(medication_agent(user_input))
        print(f"\nMedication Agent: {response}\n")

## Inspecionando Branches de Memória de Saúde

Uma das principais vantagens do AgentCore Memory Branching é a capacidade de inspecionar o histórico de conversas de cada agente de saúde de forma independente. Isso é crucial para:

**Depuração de Sistemas Multi-Agente de Saúde:**
- Veja exatamente o que cada agente de saúde discutiu com o paciente
- Identifique qual agente tratou qual consulta médica
- Rastreie o fluxo de informações do paciente através do sistema de saúde

**Entendendo a Coordenação de Agentes de Saúde:**
- Verifique se os agentes mantiveram contextos médicos separados
- Confirme que não ocorreram conflitos de dados de pacientes durante execução concorrente
- Audite a linha do tempo das interações dos agentes de saúde
- Garanta conformidade com HIPAA através do rastreamento isolado de conversas

**Benefícios Específicos para Saúde:**
- **Agente de Sinistros**: Rastreie todas as discussões sobre seguros e faturamento
- **Agente de Dados Demográficos**: Monitore atualizações e mudanças nas informações do paciente
- **Agente de Medicamentos**: Audite todas as conversas sobre prescrições e medicamentos
- **Agente Supervisor**: Revise decisões de roteamento e triagem de pacientes

Vamos explorar as branches de saúde que foram criadas durante nossa consulta com o paciente:

In [ ]:
print("\n=== Viewing Healthcare Memory Branches ===")

if claims_hooks or demographics_hooks or medication_hooks:
    # Get any memory session to list branches (they all point to the same session)
    hook = (
        claims_hooks
        if claims_hooks
        else (demographics_hooks if demographics_hooks else medication_hooks)
    )
    if hook:
        memory_session = hook.get_session(actor_id=PATIENT_ID, session_id=SESSION_ID)

        # List all branches in the session
        branches = memory_session.list_branches()
        print(f"\n📊 Session has {len(branches)} branches total:")
        for branch in branches:
            events = memory_session.list_events(branch_name=branch.name)
            print(f"  - Branch: {branch.name}")
            print(f"    └─ Events: {len(events)}")
            print(f"    └─ Created: {branch.created}")

            # Print recent conversations from this branch
            if events:
                print("    └─ Recent conversations:")
                for event in events[-100:]:  # Show last 10 events
                    for payload in event.payload:
                        if "conversational" in payload:
                            role = payload["conversational"]["role"]
                            text = payload["conversational"]["content"]["text"]
                            print(f"        {role}: {text[:500]}...")

        print("\n💡 Each branch represents a different agent's memory:")
        print("  • 'main' = Supervisor agent conversations")
        print("  • 'claims_agent' = Claims assistant conversations")
        print("  • 'demographics_agent' = Demographics assistant conversations")
        print("  • 'medication_agent' = Medication assistant conversations")
else:
    print(
        "No memory hooks found. Make sure to run the cell that creates the hooks first."
    )

## Validando Memória de Longo Prazo de Saúde: Episódios e Reflexões

Vamos examinar como nosso sistema de saúde transformou conversas de curto prazo em insights estruturados de longo prazo sobre pacientes usando a **EpisodicStrategy**.

### Episódios de Saúde
Episódios capturam interações consolidadas com pacientes incluindo:
- **Contexto Clínico**: Objetivos e resultados do cuidado ao paciente
- **Coordenação de Agentes**: Como o supervisor e os agentes especialistas trabalharam juntos
- **Integração de Dados**: Recuperação e apresentação de informações do HealthLake

### Reflexões do Paciente
Reflexões fornecem insights entre episódios sobre:
- **Padrões de Cuidado**: Preferências de comunicação do paciente e necessidades recorrentes
- **Estratégias Eficazes**: Quais abordagens funcionam melhor para este paciente
- **Oportunidades de Otimização**: Áreas para melhorar o cuidado futuro

Episódios e reflexões são processados de forma assíncrona.

In [ ]:
print("=== HEALTHCARE LONG-TERM MEMORY: EPISODES & REFLECTIONS ===")
actor_id = PATIENT_ID
session_id = SESSION_ID
# Define namespaces for healthcare episodes and reflections
episode_namespace = f"healthcare/{actor_id}/{session_id}/"
reflection_namespace = f"healthcare/{actor_id}/"
print(f"\n📋 Episode namespace: {episode_namespace}")
print(f"🧠 Reflection namespace: {reflection_namespace}")

try:
    print("\n📖 HEALTHCARE EPISODES (Session-specific patient interactions)")
    episodes = client.retrieve_memories(
        memory_id=memory_id,
        namespace=episode_namespace,
        query="patient healthcare interactions",
        top_k=10,
    )
    print(f"Found {len(episodes)} healthcare episode(s)")

    for i, episode in enumerate(episodes, 1):
        print(f"\n🏥 Healthcare Episode {i}:")
        content = episode.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # Show more content for healthcare context
            print(f"   {text[:300]}..." if len(text) > 300 else f"   {text}")
        print(f"   Score: {episode.get('score', 'N/A')}")

    if not episodes:
        print("   No episodes found yet. Episodic processing happens asynchronously.")

except Exception as e:
    print(f"❌ Error retrieving healthcare episodes: {e}")

try:
    print("\n🔍 PATIENT REFLECTIONS (Cross-session healthcare insights)")
    reflections = client.retrieve_memories(
        memory_id=memory_id,
        namespace=reflection_namespace,
        query="patient care patterns and insights",
        top_k=10,
    )
    print(f"Found {len(reflections)} patient reflection(s)")

    for i, reflection in enumerate(reflections, 1):
        print(f"\n💡 Patient Reflection {i}:")
        content = reflection.get("content", {})
        if isinstance(content, dict):
            text = content.get("text", "")
            # Show more content for healthcare insights
            print(f"   {text[:400]}..." if len(text) > 400 else f"   {text}")
        print(f"   Score: {reflection.get('score', 'N/A')}")

    if not reflections:
        print(
            "   No reflections found yet. Reflections are generated after multiple episodes."
        )

except Exception as e:
    print(f"❌ Error retrieving patient reflections: {e}")

print(
    "\n💡 TIP: Use the memory browser for interactive healthcare memory visualization"
)
print("   Episodes show individual patient consultation summaries")
print("   Reflections reveal patterns in patient care and preferences")
print(
    "\n⏱️  NOTE: Episode and reflection generation takes 10-15 minutes after conversations"
)
print("   Check back later if no episodes/reflections appear immediately")

## Resumo

### O Que Construímos:
1. **Agente Supervisor** - Orquestra na branch main
2. **Agente de Sinistros** - Lida com seguros na branch claims_agent
3. **Agente de Dados Demográficos** - Gerencia informações do paciente na branch demographics_agent
4. **Agente de Medicamentos** - Lida com medicamentos na branch medication_agent

### Arquitetura de Memória:
- **Curto prazo**: Cada agente possui branch isolada
- **Episódios**: Armazenados por sessão `healthcare/{actorId}/{sessionId}/`
- **Reflexões**: Compartilhadas entre todas as sessões `healthcare/{actorId}/`

### Benefícios:
- ✅ Agentes não interferem nas conversas uns dos outros
- ✅ Todos os agentes contribuem para a memória de longo prazo da mesma sessão
- ✅ Padrões aprendidos (reflexões) compartilhados entre todas as sessões do paciente
- ✅ Histórico completo de conversas mantido por agente

## Limpeza (Opcional)

Execute esta célula para excluir a memória e a role IAM criadas neste tutorial.

In [ ]:
# import boto3

# print("Cleanup Options:")
# delete_memory = input("Delete memory? (yes/no): ").strip().lower()
# if delete_memory == 'yes':
#   try:
#       print(f"Deleting memory: {memory_id}")
#       client.delete_memory_and_wait(memory_id=memory_id)
#       print("Memory deleted")
#   except Exception as e:
#       print(f"Error deleting memory: {e}")
# else:
#   print(f"Memory preserved: {memory_id}")

# delete_healthlake = input("Delete HealthLake datastore? (yes/no): ").strip().lower()
# if delete_healthlake == 'yes':
#     try:
#         print(f"Deleting HealthLake datastore: {DATASTORE_ID}")
#         healthlake_client.delete_fhir_datastore(DatastoreId=DATASTORE_ID)
#         print("HealthLake datastore deletion initiated")
#     except Exception as e:
#         print(f"Error deleting HealthLake datastore: {e}")
# else:
#     print(f"HealthLake datastore preserved: {DATASTORE_ID}")

# print("Cleanup complete")